# Phase 3: Our own experiment (ablation + baseline comparison)

**Question:** Autoformer makes two changes to the Transformer: (1) series decomposition inside every layer and (2) Auto-Correlation
instead of self-attention. Which one actually reduces the forecast error, and is Autoformer better than a plain Transformer and a
classical statistical model (ARIMA) on the same data?

| Model | Decomposition | Auto-Correlation | Source |
|---|---|---|---|
| Autoformer | yes | yes | official code (results reused from Phase 2 when available) |
| Autoformer_NoDecomp | **no** | yes | **ours**: `models/Autoformer_NoDecomp.py` (cell 2) |
| Autoformer_FullAttn | yes | **no** (standard self-attention) | **ours**: `models/Autoformer_FullAttn.py` (cell 2) |
| Transformer | no | no | official code (`models/Transformer.py`) |
| ARIMA(1,1,1) | - | - | **ours**, statsmodels, cell 9 |

Everything else is identical to Phase 2: same data split and scaling, input 96, same training settings (Adam 1e-4, batch 32,
up to 10 epochs, patience 3), same metrics (MSE/MAE on standardised data). Only one thing changes per model, so a difference in
error can be traced to that component.

**Run it in Colab with a GPU** (Runtime -> Change runtime type -> T4 GPU), top to bottom.

In [ ]:
# 1. Check the GPU
!nvidia-smi -L || echo "No GPU: Runtime -> Change runtime type -> T4 GPU"
import torch; print("torch", torch.__version__, "| CUDA available:", torch.cuda.is_available())

In [ ]:
# 2. Official code + Phase 2 patches + our two ablation models
import os, re, subprocess
os.chdir("/content")
if not os.path.exists("/content/Autoformer"):
    !git clone -q https://github.com/thuml/Autoformer.git
os.chdir("/content/Autoformer")
print("Official Autoformer commit:", subprocess.run(["git", "rev-parse", "HEAD"], capture_output=True, text=True).stdout.strip())
!pip -q install reformer_pytorch

def patch(path, old, new):
    s = open(path).read()
    if new in s: return
    assert old in s, f"pattern not found in {path}"
    open(path, "w").write(s.replace(old, new))

# Same compatibility patches as Phase 2 (NumPy 2, pandas >= 2.2) and the seed taken from the SEED variable
patch("utils/tools.py", "np.Inf", "np.inf")
patch("utils/timefeatures.py", "    offset = to_offset(freq_str)",
      "    offset = to_offset({'t': 'min', 'T': 'min'}.get(freq_str, freq_str))")
patch("data_provider/data_loader.py", "df_stamp.drop(['date'], 1)", "df_stamp.drop(columns=['date'])")
s = open("data_provider/data_loader.py").read()
s = s.replace("df_stamp.date.apply(lambda row: row.weekday(), 1)", "df_stamp.date.dt.weekday")
s = re.sub(r"df_stamp\.date\.apply\(lambda row: row\.(month|day|hour|minute), 1\)", r"df_stamp.date.dt.\1", s)
open("data_provider/data_loader.py", "w").write(s)
patch("run.py", "    fix_seed = 2021", "    fix_seed = int(os.environ.get('SEED', 2021))")

# Ablation 1: no decomposition. Every series_decomp block passes its input through unchanged and returns a zero trend,
# the seasonal-specific layer norm becomes a standard LayerNorm, and the decoder starts from the raw last 48 steps (no trend/mean init).
open("models/Autoformer_NoDecomp.py", "w").write("""
import torch
import torch.nn as nn
from models import Autoformer
from layers.Autoformer_EncDec import series_decomp, my_Layernorm


class NoDecomp(nn.Module):
    # Replaces series_decomp: everything stays in the seasonal path, the trend is always zero.
    def forward(self, x):
        return x, torch.zeros_like(x)


class Model(Autoformer.Model):
    # Autoformer without series decomposition (Auto-Correlation kept).
    def __init__(self, configs):
        super().__init__(configs)
        for module in list(self.modules()):
            for name, child in list(module.named_children()):
                if isinstance(child, series_decomp):
                    setattr(module, name, NoDecomp())
                elif isinstance(child, my_Layernorm):
                    setattr(module, name, nn.LayerNorm(configs.d_model))

    def forward(self, x_enc, x_mark_enc, x_dec, x_mark_dec,
                enc_self_mask=None, dec_self_mask=None, dec_enc_mask=None):
        zeros = torch.zeros([x_dec.shape[0], self.pred_len, x_dec.shape[2]], device=x_enc.device)
        seasonal_init = torch.cat([x_enc[:, -self.label_len:, :], zeros], dim=1)   # raw last 48 steps + zeros
        trend_init = torch.zeros_like(seasonal_init)                                # no trend path
        enc_out = self.enc_embedding(x_enc, x_mark_enc)
        enc_out, attns = self.encoder(enc_out, attn_mask=enc_self_mask)
        dec_out = self.dec_embedding(seasonal_init, x_mark_dec)
        seasonal_part, trend_part = self.decoder(dec_out, enc_out, x_mask=dec_self_mask, cross_mask=dec_enc_mask,
                                                 trend=trend_init)
        dec_out = trend_part + seasonal_part
        if self.output_attention:
            return dec_out[:, -self.pred_len:, :], attns
        return dec_out[:, -self.pred_len:, :]
""")

# Ablation 2: Auto-Correlation replaced by standard full self-attention (non-causal, like Autoformer's own layers); decomposition kept.
open("models/Autoformer_FullAttn.py", "w").write("""
from models import Autoformer
from layers.AutoCorrelation import AutoCorrelationLayer
from layers.SelfAttention_Family import FullAttention, AttentionLayer


class Model(Autoformer.Model):
    # Autoformer with every AutoCorrelationLayer swapped for a standard multi-head self-attention layer.
    def __init__(self, configs):
        super().__init__(configs)
        for module in list(self.modules()):
            for name, child in list(module.named_children()):
                if isinstance(child, AutoCorrelationLayer):
                    setattr(module, name, AttentionLayer(
                        FullAttention(False, configs.factor, attention_dropout=configs.dropout, output_attention=False),
                        configs.d_model, configs.n_heads))
""")

# Register the two new models so run.py --model Autoformer_NoDecomp / Autoformer_FullAttn works
patch("exp/exp_main.py", "from models import Informer, Autoformer, Transformer, Reformer",
      "from models import Informer, Autoformer, Transformer, Reformer, Autoformer_NoDecomp, Autoformer_FullAttn")
patch("exp/exp_main.py", "            'Reformer': Reformer,\n",
      "            'Reformer': Reformer,\n            'Autoformer_NoDecomp': Autoformer_NoDecomp,\n            'Autoformer_FullAttn': Autoformer_FullAttn,\n")
print("Patches applied and ablation models registered.")

## Data and storage
Results are saved to Google Drive (`MyDrive/Autoformer/runs_phase3`) so a disconnect loses nothing.
When asked, click **Connect to Google Drive**, choose your account and **tick every permission box**.
If Drive is not connected, everything is kept in `/content` instead (download it with the last cell before closing).

The notebook looks for `ETTm2.csv` and `exchange_rate.csv` in `MyDrive/Autoformer/data`, `/content/data` or
`/content/Autoformer/data`; if it finds neither, it asks you to upload them.
To reuse your Phase 2 Autoformer results (and save about 2 hours), put Phase 2's `results.csv` in `MyDrive/Autoformer/runs`
or upload it to `/content` (Files panel -> upload).

In [ ]:
# 3. Storage, data and Phase 2 results
import glob, shutil
USE_DRIVE = True
DRIVE_DIR = "/content/drive/MyDrive/Autoformer"
drive_ok = False
if USE_DRIVE:
    try:
        from google.colab import drive
        drive.mount("/content/drive")
        drive_ok = True
    except Exception as e:
        print("Google Drive not connected (", e, ") -> saving in /content. Download results with the last cell.")
OUT_DIR = f"{DRIVE_DIR}/runs_phase3" if drive_ok else "/content/runs_phase3"
os.makedirs(OUT_DIR, exist_ok=True)

NEEDED = ["ETTm2.csv", "exchange_rate.csv"]
os.makedirs("dataset", exist_ok=True)
for folder in [f"{DRIVE_DIR}/data", "/content/data", "/content/Autoformer/data", "/content"]:
    for f in NEEDED:
        if os.path.exists(f"{folder}/{f}") and not os.path.exists(f"dataset/{f}"):
            shutil.copy(f"{folder}/{f}", "dataset/")
missing = [f for f in NEEDED if not os.path.exists(f"dataset/{f}")]
if missing:
    print("Please upload:", missing)
    from google.colab import files
    for name, data in files.upload().items():
        open(f"dataset/{name}", "wb").write(data)
print("dataset/ holds:", sorted(os.listdir("dataset")))

PHASE2 = None
for f in [f"{DRIVE_DIR}/runs/results.csv", "/content/results.csv", "/content/runs/results.csv"]:
    if os.path.exists(f):
        PHASE2 = f; break
print("Phase 2 results:", PHASE2 or "not found (Autoformer will be trained here as well)")
print("Saving to:", OUT_DIR)

## Settings
* `MODELS`: the models trained in this notebook. Autoformer is added automatically for any dataset/horizon missing from Phase 2.
* `HORIZONS`: Exchange-Rate is cheap (minutes per model), so all four horizons. ETTm2 takes 20 to 55 minutes per model and horizon on a T4,
  so the default is 96 and 336 (about 45 minutes per model). Add 192 and 720 if you have the GPU time.
* Exchange-Rate horizons 192 and 336 are trained for 1 epoch for **every** model, as in the official script and in Phase 2,
  so all models are compared under the same rule.
* `QUICK = True` runs tiny models for 1 epoch to check the whole notebook in a few minutes. Use `False` for the real experiment.

In [ ]:
# 4. Settings
QUICK = False
RUNS = 1
MODELS = ["Autoformer_NoDecomp", "Autoformer_FullAttn", "Transformer"]
HORIZONS = {"Exchange": [96, 192, 336, 720], "ETTm2": [96, 336]}
RUN_ORDER = ["Exchange", "ETTm2"]          # Exchange first: it finishes in minutes
FOLLOW_OFFICIAL_EPOCHS = True

DATASETS = {
    "ETTm2":    dict(data="ETTm2",  data_path="ETTm2.csv",         freq="t", factor=1, n_vars=7),
    "Exchange": dict(data="custom", data_path="exchange_rate.csv", freq="h", factor=3, n_vars=8),
}

In [ ]:
# 5. Sanity check: build every model, count parameters and run one forward pass (catches errors in seconds)
import argparse, sys, torch, numpy as np, pandas as pd
sys.path.insert(0, os.getcwd())                          # make the repo's packages importable
import models.Autoformer_NoDecomp, models.Autoformer_FullAttn
from models import Autoformer, Transformer

def model_args(name, H):
    cfg = DATASETS[name]
    small = dict(d_model=16, d_ff=32, n_heads=2) if QUICK else dict(d_model=512, d_ff=2048, n_heads=8)
    return argparse.Namespace(seq_len=96, label_len=48, pred_len=H, enc_in=cfg["n_vars"], dec_in=cfg["n_vars"],
                              c_out=cfg["n_vars"], e_layers=2, d_layers=1, factor=cfg["factor"], moving_avg=25,
                              dropout=0.05, embed="timeF", freq=cfg["freq"], activation="gelu",
                              output_attention=False, distil=True, **small)

MODULES = {"Autoformer": Autoformer, "Autoformer_NoDecomp": models.Autoformer_NoDecomp,
           "Autoformer_FullAttn": models.Autoformer_FullAttn, "Transformer": Transformer}
n_mark = {"t": 5, "h": 4}
rows = []
for m, mod in MODULES.items():
    a = model_args("ETTm2", 96)
    net = mod.Model(a).eval()
    x = torch.randn(2, 96, 7); xm = torch.randn(2, 96, n_mark["t"])
    d = torch.zeros(2, 144, 7); dm = torch.randn(2, 144, n_mark["t"])
    with torch.no_grad():
        out = net(x, xm, d, dm)
    assert out.shape == (2, 96, 7), out.shape
    rows.append({"model": m, "parameters": sum(p.numel() for p in net.parameters()), "output": tuple(out.shape)})
PARAMS = pd.DataFrame(rows).set_index("model")
display(PARAMS)

In [ ]:
# 6. Runner: one call = one training + test run of one model
import sys, time

RESULTS_CSV = f"{OUT_DIR}/results_phase3{'_quick' if QUICK else ''}.csv"

def load_results():
    return pd.read_csv(RESULTS_CSV) if os.path.exists(RESULTS_CSV) else pd.DataFrame()

def run_one(model, name, H, run):
    cfg = DATASETS[name]
    des = f"{'quick' if QUICK else 'Exp'}_run{run}"
    model_id = f"{name}_96_{H}_{model}"
    args = ["--is_training", "1", "--model", model, "--model_id", model_id,
            "--root_path", "./dataset/", "--data", cfg["data"], "--data_path", cfg["data_path"],
            "--features", "M", "--seq_len", "96", "--label_len", "48", "--pred_len", str(H),
            "--e_layers", "2", "--d_layers", "1", "--factor", str(cfg["factor"]), "--freq", cfg["freq"],
            "--enc_in", str(cfg["n_vars"]), "--dec_in", str(cfg["n_vars"]), "--c_out", str(cfg["n_vars"]),
            "--des", des, "--itr", "1", "--num_workers", "2"]
    if QUICK:
        args += ["--d_model", "16", "--d_ff", "32", "--n_heads", "2", "--train_epochs", "1"]
    elif FOLLOW_OFFICIAL_EPOCHS and name == "Exchange" and H in (192, 336):
        args += ["--train_epochs", "1"]
    env = dict(os.environ, SEED=str(2021 + run))
    t0 = time.time()
    p = subprocess.Popen([sys.executable, "-u", "run.py"] + args, env=env,
                         stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    log = []
    for line in p.stdout:
        log.append(line)
        if any(k in line for k in ("Epoch:", "Early stopping", "mse:")):
            print("   ", line.rstrip())
    p.wait()
    if p.returncode != 0:
        print("".join(log[-30:]))
        raise RuntimeError(f"run.py failed for {model_id} run {run} (last log lines above)")
    folder = glob.glob(f"results/{model_id}_{model}_*_{des}_0")[0]
    mae, mse, rmse, mape, mspe = np.load(f"{folder}/metrics.npy")
    setting = os.path.basename(folder)
    os.makedirs(f"{OUT_DIR}/checkpoints", exist_ok=True)
    shutil.copy(f"checkpoints/{setting}/checkpoint.pth", f"{OUT_DIR}/checkpoints/{setting}.pth")
    for f in ("pred.npy", "true.npy"):
        os.remove(f"{folder}/{f}")                      # large and not needed here
    return dict(model=model, dataset=name, horizon=H, run=run, mse=float(mse), mae=float(mae),
                minutes=round((time.time() - t0) / 60, 1), setting=setting)

In [ ]:
# 7. Which runs are needed: our models everywhere, Autoformer only where Phase 2 has no result
phase2 = pd.read_csv(PHASE2) if (PHASE2 and not QUICK) else pd.DataFrame(columns=["dataset", "horizon", "run"])
plan = []
for name in RUN_ORDER:
    for H in HORIZONS[name]:
        for run in range(RUNS):
            have_af = ((phase2.dataset == name) & (phase2.horizon == H) & (phase2.run == run)).any()
            for m in (MODELS if have_af else ["Autoformer"] + MODELS):
                plan.append((m, name, H, run))
print(len(plan), "training runs planned")
for p_ in plan: print("  ", *p_)

In [ ]:
# 8. Train (safe to re-run after a disconnect: finished runs are skipped)
for model, name, H, run in plan:
    done = load_results()
    if len(done) and ((done.model == model) & (done.dataset == name) & (done.horizon == H) & (done.run == run)).any():
        continue
    print(f"== {model}  {name}  horizon {H}  run {run}")
    row = run_one(model, name, H, run)
    pd.concat([done, pd.DataFrame([row])]).to_csv(RESULTS_CSV, index=False)
    print(f"   MSE {row['mse']:.3f}  MAE {row['mae']:.3f}  ({row['minutes']} min)")
display(load_results())

## ARIMA baseline (proposal: "a simpler statistical baseline")
ARIMA(1,1,1) is fitted separately to each variable of each test window, on the same 96 scaled input steps the deep models see,
and forecasts up to the longest horizon. Fitting thousands of windows is slow, so ARIMA is scored on a **regular sample** of test windows
(every 100th for ETTm2, every 10th for Exchange). "Last value" is scored on the same sample so the two can be compared directly;
the deep models are scored on all windows. Report this sampling in the write-up.

In [ ]:
# 9. ARIMA(1,1,1) on a sample of test windows
import warnings
from statsmodels.tsa.arima.model import ARIMA
ARIMA_EVERY = {"ETTm2": 100, "Exchange": 10}
if QUICK: ARIMA_EVERY = {"ETTm2": 2000, "Exchange": 200}

def split_sizes(name, n):
    if name == "ETTm2":
        return 12 * 30 * 24 * 4, 4 * 30 * 24 * 4, 4 * 30 * 24 * 4
    n_train, n_test = int(n * 0.7), int(n * 0.2)
    return n_train, n - n_train - n_test, n_test

def scaled_series(name):
    df = pd.read_csv(f"dataset/{DATASETS[name]['data_path']}").drop(columns="date")
    n_train, n_val, n_test = split_sizes(name, len(df))
    x = df.values[: n_train + n_val + n_test]
    mu, sd = x[:n_train].mean(0), x[:n_train].std(0)
    return (x - mu) / sd, n_train + n_val, n_test

ARIMA_CSV = f"{OUT_DIR}/arima{'_quick' if QUICK else ''}.csv"
if os.path.exists(ARIMA_CSV):
    arima_res = pd.read_csv(ARIMA_CSV); print("ARIMA results loaded from", ARIMA_CSV)
else:
    rows = []
    for name in RUN_ORDER:
        sc, start, n_test = scaled_series(name)
        Hmax = max(HORIZONS[name])
        idx = list(range(0, n_test - Hmax + 1, ARIMA_EVERY[name]))
        P = np.zeros((len(idx), Hmax, sc.shape[1])); T = np.zeros_like(P); L = np.zeros_like(P)
        t0 = time.time()
        for a, i in enumerate(idx):
            for v in range(sc.shape[1]):
                y = sc[start + i - 96:start + i, v]
                try:
                    with warnings.catch_warnings():
                        warnings.simplefilter("ignore")
                        P[a, :, v] = ARIMA(y, order=(1, 1, 1)).fit().forecast(Hmax)
                except Exception:
                    P[a, :, v] = y[-1]                      # rare fitting failure: fall back to the last value
                L[a, :, v] = y[-1]
                T[a, :, v] = sc[start + i:start + i + Hmax, v]
        print(f"{name}: ARIMA fitted on {len(idx)} windows x {sc.shape[1]} variables in {(time.time() - t0) / 60:.1f} min")
        for H in HORIZONS[name]:
            rows.append({"dataset": name, "horizon": H, "windows": len(idx),
                         "ARIMA MSE": np.mean((P[:, :H] - T[:, :H]) ** 2), "ARIMA MAE": np.mean(np.abs(P[:, :H] - T[:, :H])),
                         "last value MSE (same windows)": np.mean((L[:, :H] - T[:, :H]) ** 2),
                         "last value MAE (same windows)": np.mean(np.abs(L[:, :H] - T[:, :H]))})
    arima_res = pd.DataFrame(rows); arima_res.to_csv(ARIMA_CSV, index=False)
display(arima_res.round(3))

In [ ]:
# 10. Results: one table per metric, models side by side, plus the change relative to Autoformer
from numpy.lib.stride_tricks import sliding_window_view

def last_value(name, H):                                  # on ALL test windows, like the deep models
    sc, start, n_test = scaled_series(name)
    w = sliding_window_view(sc[start - 96:start + n_test], 96 + H, axis=0)
    err = w[:, :, 95:96] - w[:, :, 96:]
    return float(np.mean(err ** 2)), float(np.mean(np.abs(err)))

res = load_results()
if not QUICK and PHASE2:
    af = pd.read_csv(PHASE2).assign(model="Autoformer")
    res = pd.concat([af[af.dataset.isin(RUN_ORDER)], res])   # Autoformer is only trained here where Phase 2 had no result
res = res[[h in HORIZONS.get(d, []) for d, h in zip(res.dataset, res.horizon)]]
order = [m for m in ["Autoformer", "Autoformer_NoDecomp", "Autoformer_FullAttn", "Transformer"] if m in set(res.model)]

tables = {}
for metric in ["mse", "mae"]:
    t = res.groupby(["dataset", "horizon", "model"])[metric].mean().unstack("model")[order]
    t["last value"] = [last_value(d, h)[0 if metric == "mse" else 1] for d, h in t.index]
    a = arima_res.set_index(["dataset", "horizon"])
    t["ARIMA (sample)"] = [a.loc[(d, h), f"ARIMA {metric.upper()}"] if (d, h) in a.index else np.nan for d, h in t.index]
    tables[metric] = t
    print(f"\n{metric.upper()} (lower is better)")
    display(t.round(3))
    if "Autoformer" in t:
        change = (t[[c for c in order if c != "Autoformer"]].div(t["Autoformer"], axis=0) - 1) * 100
        print(f"{metric.upper()} change vs Autoformer, % (positive = worse than Autoformer)")
        display(change.round(1))
    t.to_csv(f"{OUT_DIR}/table_{metric}{'_quick' if QUICK else ''}.csv")

In [ ]:
# 11. Plot: MSE against horizon for every model
import matplotlib.pyplot as plt
t = tables["mse"]
fig, ax = plt.subplots(1, len(RUN_ORDER), figsize=(7 * len(RUN_ORDER), 4))
ax = np.atleast_1d(ax)
for a, name in zip(ax, RUN_ORDER):
    sub = t.loc[name]
    for col in sub.columns:
        style = "--" if col in ("last value", "ARIMA (sample)") else "-"
        a.plot(sub.index, sub[col], style, marker="o", label=col)
    a.set_title(f"{name}: test MSE by horizon"); a.set_xlabel("horizon (steps)"); a.set_ylabel("MSE (scaled)")
    a.set_xticks(sub.index)
ax[-1].legend(bbox_to_anchor=(1.02, 1), loc="upper left")
plt.tight_layout(); plt.savefig(f"{OUT_DIR}/mse_by_horizon.png", dpi=150, bbox_inches="tight"); plt.show()

In [ ]:
# 12. Answer the question with the numbers (average change in MSE vs Autoformer over the horizons run)
t = tables["mse"]
if "Autoformer" in t:
    for name in RUN_ORDER:
        sub = t.loc[name]
        print(f"\n{name}:")
        for m, what in [("Autoformer_NoDecomp", "removing decomposition"),
                        ("Autoformer_FullAttn", "replacing Auto-Correlation with self-attention"),
                        ("Transformer", "using a plain Transformer"),
                        ("last value", "repeating the last value"),
                        ("ARIMA (sample)", "ARIMA(1,1,1), sampled windows")]:
            if m in sub:
                ch = ((sub[m] / sub["Autoformer"] - 1) * 100).mean()
                print(f"  {what:50s}: MSE {'+' if ch >= 0 else ''}{ch:.1f}% on average vs Autoformer")
    print("\nRead it like this: the component whose removal raises the error most contributes most on that dataset.")
    print("Check the per-horizon table above too: an average can hide a horizon where the ranking flips.")
print("\nParameters per model (ETTm2 configuration):"); display(PARAMS[["parameters"]])

In [ ]:
# 13. Download everything (needed if Drive was not connected; harmless otherwise)
zip_path = shutil.make_archive("/content/runs_phase3", "zip", root_dir=os.path.dirname(OUT_DIR), base_dir=os.path.basename(OUT_DIR))
print("Created", zip_path, f"({os.path.getsize(zip_path) / 1e6:.0f} MB)")
from google.colab import files; files.download(zip_path)

## What goes in the report (section 7, "Additional experiment")
* The question (top of this notebook), the five models and what changes in each (table at the top).
* The MSE and MAE tables (cell 10, saved as `table_mse.csv` / `table_mae.csv`) and the plot (cell 11, `mse_by_horizon.png`).
* The answer from cell 12, with the per-horizon caveats.
* Training minutes per model (cell 8 table) and parameter counts (cell 5): useful for the computational comparison.
* What is ours: `Autoformer_NoDecomp.py`, `Autoformer_FullAttn.py`, the ARIMA baseline, the runner and the analysis cells.
* Limits: one run per setting unless `RUNS = 3`; ETTm2 only at the horizons run; ARIMA on sampled windows with a fixed order.